## 1. Импорты

In [1]:
import gc, hashlib, os, re, shutil, time
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from pymorphy3 import MorphAnalyzer
from scipy import sparse as sp
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.util import cos_sim
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

DATA = '/home/mluser/work/avito'
E = lambda k, d: os.environ.get('AVITO_' + k, d)
I = lambda k, d: int(E(k, d))
B = lambda k: E(k, '0') == '1'

SEED, TOPK, N_BUCKETS = 42, 50, 5        # 1 item_id из 5 уходит в валидацию
MODEL = E('MODEL', 'intfloat/multilingual-e5-small')
TAG = E('TAG', MODEL.split('/')[-1])     # чекпойнт nn_<TAG>.pt; проверенный - nn_e5s.pt
SPARSE_VAL = 0.4683                      # Recall@50 чисто sparse на этом же val-сплите
SPARSE_BACKUP = f'{DATA}/answer_sparse_backup.csv'
MAX_LEN, BATCH, N_NEG = I('LEN', 128), I('BATCH', 192), I('NNEG', 2)
MAX_STEPS, BUDGET_S = I('STEPS', 1500), I('BUDGET', 1500)
LR, TAU, DESC_CHARS = float(E('LR', 1e-4)), float(E('TAU', 0.05)), I('DESC', 200)
LOAD_CKPT, WRITE_ANSWER, SMOKE = B('LOAD'), B('WRITE'), B('SMOKE')

if SMOKE:
    N_VAL, N_RANK, N_CORPUS = 800, 1500, 20000
    POOL, TITLE_POOL, CHAR_POOL, DENSE_POOL = 40, 20, 20, 20
    BATCH, MAX_STEPS, BUDGET_S = min(BATCH, 64), I('STEPS', 150), 600
else:
    N_VAL, N_RANK, N_CORPUS = 10000, 40000, 0
    POOL, TITLE_POOL, CHAR_POOL, DENSE_POOL = 400, 150, 150, 50

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = dict(device_type=DEV, dtype=torch.bfloat16, enabled=(DEV == 'cuda'))
torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = True

def hit(top_ids, rel, k=TOPK):
    """Доля релевантных объявлений в топ-k одного запроса."""
    return len(rel & set(top_ids[:k])) / len(rel)

## 2. Тексты и пакетный top-k
Фильтры приходят склейкой пар «Ключ Значение» без разделителей, поэтому всё послепервого мусорного ключа отбрасываем. `csr_topk` — точный top-k из разреженного произведениячерез `sparse-dot-topn`, без плотных матриц.

In [2]:
NOISE_KEYS = ['место оказания услуг', 'тип стоимости за услугу', 'рабочие дни', 'время для связи',
              'график работы', 'опыт работы', 'гарантия', 'как вы работаете', 'работаете с', 'бригада',
              'предоплата', 'работа по договору', 'стоимость', 'цена', 'продолжительность', 'рейтинг',
              'онлайн-запись', 'способ', 'минимальный', 'адрес', 'режим работы', 'срок', 'договор',
              'звонок', 'обратная связь', 'выезд', 'акция', 'скидка', 'абонемент', 'разовое']
_NON = re.compile(r'[^0-9a-zа-я]+')
_morph, _stem_cache = MorphAnalyzer(), {}


def _s(x):
    return '' if x is None or (isinstance(x, float) and np.isnan(x)) else str(x)


def clean(s):
    s = _s(s).lower().replace('\xa0', ' ')
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', s, flags=re.U)).strip()


def params_head(s, limit=200):
    """Информативная часть фильтров: всё до первого мусорного ключа."""
    s, j = clean(s), len(s)
    for k in NOISE_KEYS:
        p = s.find(' ' + k)
        if p >= 0:
            j = min(j, p)
    return s[:j][:limit]


def stem(w):
    """Срез нормальной формы до 4 символов склеивает больше вариантов."""
    if w not in _stem_cache:
        p = _morph.parse(w)
        _stem_cache[w] = (p[0].normal_form if p else w)[:4]
    return _stem_cache[w]


def prep(s, do_stem=True, max_tokens=None):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return []
    toks = _NON.sub(' ', _s(s).lower().replace('ё', 'е').replace('\n', ' ').replace('\t', ' ')).split()
    if do_stem:
        toks = [stem(t) if (t.isalpha() and len(t) > 3) else t for t in toks]
    return toks[:max_tokens] if max_tokens else toks


def prep_str(s, do_stem=True, max_tokens=None):
    return ' '.join(prep(s, do_stem, max_tokens))


def item_text(title, params, desc=''):
    """Заголовок дублируется: в sparse-пайплайне title x2 давал лучший канал."""
    t = clean(title)
    return ('passage: ' + t + ' ' + t + ' ' + params_head(params) + ' '
            + clean(desc)[:DESC_CHARS]).strip()


def csr_topk(Q, XT, k, chunk=1024, n_threads=-1):
    """Q (n,V) csr @ XT (V,m) -> (idx, score) формы (n,k); -1 = «кандидата нет»."""
    n = Q.shape[0]
    oi = np.full((n, k), -1, np.int32)
    os_ = np.zeros((n, k), np.float32)
    for st in range(0, n, chunk):
        C = sp_matmul_topn(Q[st:st + chunk], XT, top_n=k, sort=True, n_threads=n_threads)
        cnt = np.diff(C.indptr)                                    # сколько кандидатов у строки
        pos = np.arange(C.nnz) - np.repeat(C.indptr[:-1], cnt)     # позиция внутри строки
        keep = pos < k
        r = np.repeat(np.arange(C.shape[0], dtype=np.int64), cnt)[keep]
        oi[st + r, pos[keep]], os_[st + r, pos[keep]] = C.indices[keep], C.data[keep]
    return oi, os_

## 3. Данные, сплит, тексты
В валидацию уходят группы, у которых все релевантные объявления попали в бакет 0по md5 от `item_id`, — иначе метрика завышается. Строим `qtext` для энкодера и`qtext_idx`/`qtoks` для TF-IDF и покрытия.

In [4]:
ITEM_COLS = ['item_id', 'item_microcat_id', 'item_title_raw', 'item_infm_params_text',
             'item_description_raw', 'item_price', 'item_rating', 'item_rating_reviews_count']
train = pd.read_parquet(f'{DATA}/train.parquet')
bq = pd.read_parquet(f'{DATA}/benchmark_queries.parquet')
bi = pd.read_parquet(f'{DATA}/benchmark_items.parquet', columns=ITEM_COLS)
train_items = train.drop_duplicates('item_id').reset_index(drop=True)
assert bi.item_id.is_unique and bq.query_id.is_unique

train['qkey'] = train.search_query.astype(str) + '\u0001' + train.search_location_id.astype(str)
_g = train.groupby('qkey', sort=False)
qmeta = _g.agg(search_query=('search_query', 'first'), q_params=('search_infm_params_text', 'first'),
               loc_id=('search_location_id', 'first')).reset_index()
truth = _g.item_id.apply(set).to_dict()          # qkey -> {релевантные item_id}
del _g, train
gc.collect()

bucket_of = lambda iid: int(hashlib.md5(iid.encode()).hexdigest()[:8], 16) % N_BUCKETS
item_b = {i: bucket_of(i) for t in truth.values() for i in t}
val_keys = [k for k, t in truth.items() if all(item_b[i] == 0 for i in t)]
trn_keys = [k for k, t in truth.items() if all(item_b[i] != 0 for i in t)]
r2 = np.random.default_rng(SEED)
val_sel = [val_keys[i] for i in r2.permutation(len(val_keys))[:N_VAL]]
rank_sel = [trn_keys[i] for i in r2.permutation(len(trn_keys))[:N_RANK]]

# у qmeta параметры лежат в q_params, у bq - в search_infm_params_text: колонки разные
for df, pcol in ((qmeta, 'q_params'), (bq, 'search_infm_params_text')):
    df['qtext_idx'] = [' '.join(prep(a, True, 30) + prep(a, False, 30)) for a in df.search_query]
    df['qtoks'] = [prep(a, True, 30) + prep(a, False, 30) + prep(b, False, 40)
                   for a, b in zip(df.search_query, df[pcol])]
    df['qtext'] = ['query: ' + clean(a) + ' ' + params_head(b)
                   for a, b in zip(df.search_query, df[pcol])]
# микрокатегория запроса берётся через p2m (нормализованные параметры -> микрокат. объявления)
_pk = lambda p: ' '.join(sorted(prep(p, False, 20)))
p2m = {}
for p, c in zip(train_items.item_infm_params_text, train_items.item_microcat_id):
    p2m.setdefault(_pk(p), c)
for df, pcol in ((qmeta, 'q_params'), (bq, 'search_infm_params_text')):
    df['qmicro'] = [p2m.get(_pk(p), -1) for p in df[pcol]]
del p2m
bq['loc_id'] = bq.search_location_id

val_rows = qmeta[qmeta.qkey.isin(val_sel)].reset_index(drop=True)
rank_rows = qmeta[qmeta.qkey.isin(rank_sel)].reset_index(drop=True)
enc_rows = qmeta[qmeta.qkey.isin(trn_keys)][['qkey', 'qtext']]     # обучение энкодера
for df in (train_items, bi):
    df['itext'] = [item_text(t, p, d) for t, p, d in
                   zip(df.item_title_raw, df.item_infm_params_text, df.item_description_raw)]
    df['stext'] = [prep_str(t, True, 40) for t in df.item_title_raw]
    df['ptext'] = [prep_str(p, False, 60) for p in df.item_infm_params_text]

if SMOKE:                                   # маленький корпус, но со всеми нужными позитивами
    id2row = {v: i for i, v in enumerate(train_items.item_id)}
    must = {id2row[i] for kk in list(val_rows.qkey) + list(rank_rows.qkey)
            for i in truth[kk] if i in id2row}
    must = np.fromiter(must, np.int64)
    rest = np.setdiff1d(np.arange(len(train_items)), must)
    extra = rng.choice(rest, size=max(N_CORPUS - len(must), 0), replace=False)
    keep = np.sort(np.concatenate([must, extra]))
    train_items = train_items.iloc[keep].reset_index(drop=True)
    log(f'SMOKE: корпус сокращён до {len(train_items)} объявлений')
    del id2row, must, rest, extra, keep
del qmeta
gc.collect()

0

## 4. Покрытие токенов
`Coverage` держит бинарную матрицу «объект × токен»; сколько общих токенов у запросаи объявления — это ненулевые ячейки sparse-произведения, то есть `nnz`.

In [5]:
class Coverage:
    def __init__(self, token_lists):
        vocab, rows, cols = {}, [], []
        for i, toks in enumerate(token_lists):
            for t in set(toks):
                rows.append(i)
                cols.append(vocab.setdefault(t, len(vocab)))
        self.vocab = vocab
        self.M = sp.csr_matrix((np.ones(len(rows), np.int8), (rows, cols)),
                               shape=(len(token_lists), max(len(vocab), 1)))

    def ids_of(self, toks):
        v = self.vocab
        return np.fromiter({v[t] for t in set(toks) if t in v}, np.int32, -1)

    def batch(self, qids_list, rows, row_group, sub=256):
        """qids_list - id-ы токенов запросов (из ids_of), rows - строки корпуса-кандидаты."""
        out = np.zeros(len(rows), np.float32)
        for s0 in range(0, len(qids_list), sub):
            s1 = min(s0 + sub, len(qids_list))
            sel = (row_group >= s0) & (row_group < s1)
            rr = rows[sel]
            if not len(rr):
                continue
            qlens = np.fromiter((len(q) for q in qids_list[s0:s1]), np.int64, s1 - s0)
            q = np.concatenate(qids_list[s0:s1])
            if not len(q):
                continue
            owner = np.repeat(np.arange(s1 - s0, dtype=np.int64), qlens)
            Q = sp.csr_matrix((np.ones(len(q), np.int8), (owner, q)),
                              shape=(s1 - s0, self.M.shape[1]))
            prod = (Q @ self.M[rr].T).toarray()             # (запросы x кандидаты подбатча)
            out[sel] = prod[(row_group[sel] - s0), np.arange(len(rr))].astype(np.float32)
        return out

## 5. Корпус с предрассчитанными признаками
Два TF-IDF-индекса (весь текст и заголовок), покрытие по заголовку и фильтрам,свойства объявления. Индексы сразу хранятся транспонированными — именно они нужныдля top-k.

In [6]:
class Corpus:
    def __init__(self, df, tag):
        self.n, self.tag = len(df), tag
        self.ids = df.item_id.values.copy()
        self.id2row = {v: i for i, v in enumerate(self.ids)}
        price = df.item_price.astype('float64').fillna(-1).values
        self.log_price = np.log1p(np.clip(price, 0, None)).astype(np.float32)
        self.rating = df.item_rating.fillna(0).values.astype(np.float32)
        self.log_rev = np.log1p(df.item_rating_reviews_count.fillna(0).values).astype(np.float32)

        def tf(name, text, **kw):
            v = TfidfVectorizer(sublinear_tf=True, dtype=np.float32, **kw)
            X = v.fit_transform(text)
            log(f'[{tag}] {name} -> {X.shape}, nnz={X.nnz}')
            return v, X.T.tocsr()

        self.vec_w, self.X_wT = tf('word TF-IDF', df.itext.tolist(), ngram_range=(1, 2), min_df=3)
        self.vec_t, self.X_tT = tf('title TF-IDF', df.stext.tolist(), ngram_range=(1, 2), min_df=2)
        self.vec_c, self.X_cT = tf('char TF-IDF', df.itext.tolist(), analyzer='char_wb',
                                   ngram_range=(3, 5), min_df=5, max_features=400000)
        self.cov_t, self.cov_p = Coverage(df.stext.tolist()), Coverage(df.ptext.tolist())
        # микрокатегория -> строки корпуса; запрос тянет все объявления своей категории
        self.micro = df.item_microcat_id.values
        cats = {}
        for c, r in zip(self.micro, range(self.n)):
            cats.setdefault(c, []).append(r)
        self.cat_rows = {c: np.array(v, np.int64) for c, v in cats.items()}
        self.cat_size = {c: np.log1p(len(v)) for c, v in self.cat_rows.items()}
        log(f'[{tag}] корпус готов: {self.n} объявлений')

## 6. Энкодер sentence-transformers
`SentenceTransformer` целиком: `Transformer` (веса) + `Pooling` (mean) + `Normalize`,то есть косинусное расстояние без своих строк кода. Лосс — библиотечный`MultipleNegativesRankingLoss`: позитивы первыми `B` строками, дальше негативы, `scale = 1/TAU`.

In [ ]:
def load_encoder_weights(st_model, path):
    """state_dict -> веса transformer-модели; у ранних сборок ключи начинались с 'enc.'."""
    sd = torch.load(path, map_location=DEV)
    st_model[0].auto_model.load_state_dict(
        {k[len('enc.'):] if k.startswith('enc.') else k: v for k, v in sd.items()})


model = SentenceTransformer(MODEL, device=DEV)
model.max_seq_length = MAX_LEN
TOK = model.tokenizer
CKPT = f'{DATA}/nn_{TAG}.pt'
if LOAD_CKPT and os.path.exists(CKPT):
    load_encoder_weights(model, CKPT)

id2row = {v: i for i, v in enumerate(train_items.item_id)}
QTRN = enc_rows.qtext.tolist()
pos = [np.array([id2row[i] for i in truth[kk] if i in id2row], np.int32) for kk in enc_rows.qkey]
keep = [i for i, p in enumerate(pos) if len(p)]
QTRN = [QTRN[i] for i in keep]
trn_rows = [pos[i] for i in keep]         # по одному положительному на группу
# негативы случайные: шанс случайно попасть в релевантные пренебрежимо мал
neg_pool = [rng.integers(0, len(train_items), N_NEG * 4) for _ in trn_rows]


def embed_texts(texts):
    enc = TOK(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
    return {k: v.to(DEV, non_blocking=True) for k, v in enc.items()
            if k in ('input_ids', 'attention_mask')}


loss_fn = MultipleNegativesRankingLoss(model=model, scale=1.0 / TAU)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=MAX_STEPS,
                                            pct_start=0.1, anneal_strategy='cos')


def train_step(gidx):
    nb = len(gidx)
    gpos = np.array([trn_rows[i][rng.integers(len(trn_rows[i]))] for i in gidx], np.int64)
    gneg = np.stack([neg_pool[i][rng.integers(len(neg_pool[i]), size=N_NEG)] for i in gidx])
    with torch.autocast(**AMP):
        zq = model(embed_texts([QTRN[i] for i in gidx]))['sentence_embedding']
        zi = model(embed_texts([train_items.itext.iloc[j] for j in
                                np.concatenate([gpos, gneg.ravel()])]))['sentence_embedding']
    zq, zi = zq.float(), zi.float()     # логиты строго fp32: при TAU=0.05 bf16 теряет точность
    loss = loss_fn.compute_loss_from_embeddings([zq, zi], labels=None)
    acc = float((cos_sim(zq, zi).argmax(1) == torch.arange(nb, device=DEV)).float().mean())
    opt.zero_grad(set_to_none=True)
    loss.backward()
    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    return float(loss.detach()), acc, float(gn)


if LOAD_CKPT and os.path.exists(CKPT):
    pass
else:
    model.train()
    t_start, ema = time.time(), 0.0
    perm, step, cur = rng.permutation(len(trn_rows)), 0, 0
    for step in range(1, MAX_STEPS + 1):
        if cur + BATCH > len(perm):
            perm, cur = rng.permutation(len(trn_rows)), 0
        l, a, _ = train_step(perm[cur:cur + BATCH])
        cur += BATCH
        ema = 0.95 * ema + 0.05 * l
        if step % 100 == 0:
            el = time.time() - t_start
        if time.time() - t_start > BUDGET_S:
            break
    torch.save(model[0].auto_model.state_dict(), CKPT)

del opt, sched, loss_fn
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 7. Кодирование и dense-поиск
Кодируем корпус и запросы через `SentenceTransformer.encode` под `autocast` (bf16) и ищемкосинусных соседей точным matmul на GPU — на 345k×384 это секунды, ANN-библиотека не нужна.

In [ ]:
def encode(texts, bs=512, tag=''):
    model.eval()
    t0 = time.time()
    with torch.autocast(**AMP):
        v = model.encode(list(texts), batch_size=bs, convert_to_numpy=True,
                         normalize_embeddings=True, show_progress_bar=False)
    log(f'  {tag}: {len(texts)} векторов за {time.time() - t0:.0f}с')
    return v.astype(np.float16)


def search(Q, emb, k, chunk=512):
    """Точный косинусный поиск: Q (n,d) и emb (m,d) fp16 -> (n,k) строки корпуса + скоры."""
    G = torch.from_numpy(np.ascontiguousarray(emb)).to(DEV)
    rows = np.full((Q.shape[0], k), -1, np.int32)
    sc = np.zeros((Q.shape[0], k), np.float32)
    for st in range(0, Q.shape[0], chunk):
        q = torch.from_numpy(np.ascontiguousarray(Q[st:st + chunk])).to(DEV)
        s = (q @ G.T).float()                   # fp16 matmul, но topk сразу в fp32
        v, j = torch.topk(s, k, dim=1)
        rows[st:st + chunk], sc[st:st + chunk] = j.cpu().numpy(), v.cpu().numpy()
    del G
    torch.cuda.empty_cache()
    return rows, sc


def dense_candidates(qtexts, tag, k=DENSE_POOL, items_emb=None):
    """-> (list строк корпуса, list косинусов). Строки индексируют emb_items либо items_emb."""
    return [list(x) for x in search(encode(list(qtexts), tag=tag),
                                    emb_items if items_emb is None else items_emb, k)]


ITR = train_items.itext.tolist()
emb_items = encode(ITR, tag='train-items')
dva, sva = dense_candidates(val_rows.qtext.tolist(), 'val')
dra, sra = dense_candidates(rank_rows.qtext.tolist(), 'rank')
del ITR
gc.collect()

row2id = list(train_items.item_id)
cov_d = float(np.mean([bool(truth[kk] & {row2id[i] for i in r if i >= 0})
                       for kk, r in zip(val_rows.qkey, dva)]))
log(f'RECALL@50 dense в одиночку: '
    f'{np.mean([hit([row2id[i] for i in r if i >= 0], truth[kk]) for r, kk in zip(dva, val_rows.qkey)]):.4f}'
    f' | покрытие {cov_d:.4f} | среди попавших '
    f'{np.mean([hit([row2id[i] for i in r if i >= 0], truth[kk]) for r, kk in zip(dva, val_rows.qkey)]) / max(cov_d, 1e-9):.4f}')

## 8. Пул кандидатов и признаки
Пять каналов-кандидатов (word, title, char, micro, dense) сливаются в один пул по батчузапросов: ключ `запрос × строка` сортируется, дубли схлопываются, каждому кандидатупроставляются score и rank по каждому каналу. `FEATS` — 19 признаков.Покрытие пула — потолок всего пайплайна: без char- и micro-каналов word+title+denseдают 0.2330, а с ними 0.5528.

In [ ]:
N_CH = 5                                   # 0 word, 1 title, 2 char, 3 micro, 4 dense
FEATS = ['s_word', 'r_word', 's_title', 'r_title', 's_char', 'r_char', 's_max', 'mc_match',
         'mc_size_log', 'cov_title', 'n_title_match', 'cov_param',
         'log_price', 'rating', 'log_rev', 's_dense', 'r_dense', 'from_dense', 'dense_only']
NF = len(FEATS)
EMPTY = np.zeros(0, np.int64)
I_FROM_DENSE = FEATS.index('from_dense')   # X[:, I_FROM_DENSE] = «это принёс dense-канал»


def pool_for(C, qtext_idx, n_items, mc_rows, mcsc, dense_rows, dense_scores):
    """-> row_group, rows, s/r каждого из 3 каналов, gsz. Без python-цикла по запросам."""
    n = len(qtext_idx)
    if not n:
        z = np.zeros(0, np.int32)
        return (z, z) + (z, z) * N_CH + (z,)
    base = np.arange(n, dtype=np.int64) * n_items
    keys, vals, ranks, chs = [], [], [], []

    def push(rows, ch, scores=None):
        """rows: (n,k) индексы корпуса, -1 = «нет кандидата»."""
        ok = rows >= 0
        keys.append((base[:, None] + rows)[ok])
        vals.append(np.zeros(int(ok.sum()), np.float32) if scores is None
                    else np.asarray(scores, np.float32)[ok])
        ranks.append(np.tile(np.arange(rows.shape[1], dtype=np.int32), (n, 1))[ok])
        chs.append(np.full(int(ok.sum()), ch, np.int8))

    for vec, XT, k, ch in ((C.vec_w, C.X_wT, POOL, 0), (C.vec_t, C.X_tT, TITLE_POOL, 1),
                           (C.vec_c, C.X_cT, CHAR_POOL, 2)):
        ii, ss = csr_topk(vec.transform(qtext_idx), XT, k)
        push(ii.astype(np.int64), ch, ss)
    lens = np.fromiter(map(len, mc_rows), np.int64, n)         # micro: вся категория, без top-k
    if lens.sum():
        own = np.repeat(np.arange(n, dtype=np.int64), lens)
        keys.append(base[own] + np.concatenate(mc_rows))
        vals.append(np.repeat(np.asarray(mcsc, np.float32), lens))
        ranks.append(np.zeros(int(lens.sum()), np.int32))
        chs.append(np.full(int(lens.sum()), 3, np.int8))
    push(np.asarray(dense_rows, np.int64), 4, dense_scores)

    key, val = np.concatenate(keys), np.concatenate(vals)
    rank, ch = np.concatenate(ranks), np.concatenate(chs)
    del keys, vals, ranks, chs
    o = np.argsort(key, kind='stable')
    key, val, rank, ch = key[o], val[o], rank[o], ch[o]

    new = np.empty(len(key), bool)
    new[0] = True
    np.not_equal(key[1:], key[:-1], out=new[1:])
    seg, starts = np.cumsum(new) - 1, np.flatnonzero(new)
    row_group = (key[starts] // n_items).astype(np.int32)
    rows = (key[starts] % n_items).astype(np.int32)

    s_out, r_out = {}, {}
    for c in range(N_CH):
        s_c = np.zeros(len(starts), np.float32)
        r_c = np.full(len(starts), 999, np.int32)
        m = ch == c
        if m.any():
            idx = np.flatnonzero(m)
            kk = key[idx]
            st = np.flatnonzero(np.concatenate([[True], kk[1:] != kk[:-1]]))
            dest = seg[idx[st]]
            s_c[dest] = np.maximum.reduceat(val[idx], st)
            r_c[dest] = np.minimum.reduceat(rank[idx], st)
        s_out[c], r_out[c] = s_c, r_c

    gsz = np.bincount(row_group, minlength=n).astype(np.int32)
    chans = tuple(v for c in range(N_CH) for v in (s_out[c], r_out[c]))
    return (row_group, rows) + chans + (gsz,)


def featurize(C, rows, pool, qtok_lens, qids_t, qids_p, qmicro, mcsc):
    """-> (X, gsz, row_group) по всему батчу; ни одного python-цикла по группам."""
    rg, urow = pool[0], pool[1]
    (s_w, r_w, s_t, r_t, s_c, r_c, s_m, r_m, s_d, r_d), gsz = pool[2:-1], pool[-1]
    m = len(urow)
    X = np.empty((m, NF), np.float32)
    if not m:
        return X, gsz, rg
    ct, cp = C.cov_t.batch(qids_t, urow, rg), C.cov_p.batch(qids_p, urow, rg)
    tl = np.maximum(np.asarray(qtok_lens, np.int32)[rg], 1)
    X[:, 0], X[:, 1] = s_w, 1.0 / (1.0 + r_w)
    X[:, 2], X[:, 3] = s_t, 1.0 / (1.0 + r_t)
    X[:, 4], X[:, 5] = s_c, 1.0 / (1.0 + r_c)
    X[:, 6] = np.maximum(np.maximum(s_w, s_t), s_c)
    X[:, 7] = (C.micro[urow] == np.asarray(qmicro)[rg]).astype(np.float32)
    X[:, 8] = np.asarray(mcsc, np.float32)[rg]
    X[:, 9], X[:, 10], X[:, 11] = ct / tl, ct, cp / tl
    X[:, 12], X[:, 13], X[:, 14] = C.log_price[urow], C.rating[urow], C.log_rev[urow]
    in_d = r_d < 999
    X[:, 15], X[:, 16], X[:, 17] = s_d, 1.0 / (1.0 + r_d), in_d
    X[:, 18] = in_d & (r_w == 999) & (r_t == 999) & (r_c == 999)   # принёс только dense
    return X, gsz, rg

## 9. Ранкерная матрица и LambdaRank
Метрика считается ровно как в задаче — доля релевантных в топ-50, усреднённая по **всем**запросам val, включая те, где в пуле позитива нет: они дают ноль.

In [ ]:
def make_dataset(C, rows, qkey2rel=None, batch=8000):
    """-> (X, y, g, keys, row_ids). row_ids[i] = строка корпуса для X[i]."""
    t0, xs, ys, gs, rs, keys_all = time.time(), [], [], [], [], []
    for st in range(0, len(rows), batch):
        part = rows.iloc[st:st + batch]
        qt, qm = part.qtoks.values, part.qmicro.values
        mc = [C.cat_rows.get(c, EMPTY) for c in qm]
        mcsc = [C.cat_size.get(c, 0.0) for c in qm]
        pool = pool_for(C, part.qtext_idx.values, C.n, mc, mcsc,
                        np.stack(part.dr.values), np.stack(part.ds.values))
        rg, urow = pool[0], pool[1]
        X, gsz, rg = featurize(C, part, pool, [len(t) for t in qt],
                               [C.cov_t.ids_of(t) for t in qt], [C.cov_p.ids_of(t) for t in qt],
                               qm, mcsc)
        y, keep = None, np.zeros(len(part), bool)
        if qkey2rel is not None:
            y = np.zeros(len(urow), np.int8)
            gstarts = np.searchsorted(rg, np.arange(len(part) + 1))
            for r in range(len(part)):
                t = qkey2rel.get(part.qkey.iloc[r], set())
                if not t or gsz[r] == 0:
                    continue
                tr = np.fromiter((C.id2row[i] for i in t if i in C.id2row), np.int32, -1)
                a, b = int(gstarts[r]), int(gstarts[r + 1])
                hit_ = np.isin(urow[a:b], tr).astype(np.int8)
                y[a:b] = hit_
                keep[r] = bool(hit_.any())
            X, y, urow = X[keep[rg]], y[keep[rg]], urow[keep[rg]]
            keys_all.extend(part.qkey.values[keep])
        rs.append(urow)
        g = gsz[keep] if qkey2rel is not None else gsz
        if len(g):
            xs.append(X), gs.append(g)
            if y is not None:
                ys.append(y)
        del X, pool, urow
    X = np.concatenate(xs) if xs else np.zeros((0, NF), np.float32)
    y = np.concatenate(ys) if ys else np.zeros(0, np.int8)
    g = np.concatenate(gs) if gs else np.zeros(0, np.int32)
    row_ids = np.concatenate(rs) if rs else np.zeros(0, np.int32)
    assert int(g.sum()) == X.shape[0] == row_ids.shape[0]
    log(f'  {C.tag}: X={X.shape}, групп={len(g)}, доля pos={y.mean() if len(y) else 0:.4f}, '
        f'{time.time() - t0:.0f}с')
    return X, y, g, keys_all, row_ids


C_tr = Corpus(train_items, 'train-corpus')
assert (C_tr.ids == train_items.item_id.values).all(), 'плотные индексы не совпадают с корпусом'
val_rows['dr'], val_rows['ds'] = [list(x) for x in dva], [list(x) for x in sva]
rank_rows['dr'], rank_rows['ds'] = [list(x) for x in dra], [list(x) for x in sra]
Xtr, ytr, gtr, _, _ = make_dataset(C_tr, rank_rows, truth)
Xva, yva, gva, kva, ids_va = make_dataset(C_tr, val_rows, truth)
gc.collect()

PARAMS = dict(objective='lambdarank', metric='ndcg', learning_rate=0.05, num_leaves=63,
              min_data_in_leaf=200, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              lambdarank_truncation_level=60, n_estimators=300, n_jobs=16, verbose=-1)
ranker = lgb.LGBMRanker(**PARAMS)
ranker.fit(Xtr, ytr, group=gtr)
gc.collect()

offs = np.concatenate([[0], np.cumsum(gva)])
sc_all = ranker.predict(Xva)
rec, rec_base = [], []
for r, kk in enumerate(kva):
    a, b = offs[r], offs[r + 1]
    if a == b or not truth.get(kk):
        continue
    ids, f = C_tr.ids[ids_va[a:b]], Xva[a:b]
    rec.append(hit(ids[np.argsort(-sc_all[a:b], kind='stable')], truth[kk]))
    rec_base.append(hit(ids[np.lexsort((-f[:, 6], -f[:, 0]))], truth[kk]))
r_after, r_before = float(np.mean(rec)), float(np.mean(rec_base))
cov = len(kva) / max(len(val_rows), 1)      # группы без попадания make_dataset выкидывает, они дают 0
HYBRID_VAL = cov * r_after

## 10. Финальная модель, бенчмарк и answer.csv
Ранкер переобучается на train+val и предсказывает 2452 бенчмарк-запроса. `answer.csv`обновляется только если гибрид лучше sparse-решения, иначе остаётся прежний.

In [ ]:
del C_tr
gc.collect()
final = lgb.LGBMRanker(**PARAMS)
Xall, yall, gall = (np.concatenate(a) for a in ((Xtr, Xva), (ytr, yva), (gtr, gva)))
del Xtr, Xva, ytr, yva, gtr, gva
final.fit(Xall, yall, group=gall)
del Xall, yall, gall
gc.collect()

bq['dr'], bq['ds'] = dense_candidates(bq.qtext.tolist(), 'bench',
                                      items_emb=encode(bi.itext.tolist(), tag='bench-items'))
C_bm = Corpus(bi, 'bench-corpus')
bench_ids = set(bi.item_id)
Xb, _, gb, _, ids_bm = make_dataset(C_bm, bq, batch=1024)
del bi
gc.collect()

scores = final.predict(Xb)
offs = np.concatenate([[0], np.cumsum(gb)])
fallback = [C_bm.ids[c] for c in np.argsort(-C_bm.log_rev)[:300]]
predictions = []
for r in range(len(bq)):
    a, b = offs[r], offs[r + 1]
    out, seen = [], set()
    top = ids_bm[a:b][np.argsort(-scores[a:b], kind='stable')[:TOPK]] if a < b else np.zeros(0, np.int32)
    for c in top.tolist():
        v = C_bm.ids[c]
        if v not in seen:
            seen.add(v)
            out.append(v)
    if len(out) < TOPK:                       # добор популярными, чтобы слоты не пустовали
        out += [v for v in fallback if v not in seen][:TOPK - len(out)]
    predictions.append(out[:TOPK])

answer = pd.DataFrame({'query_id': bq.query_id.values,
                       'answer': [' '.join(p) for p in predictions]})
answer.to_csv(f'{DATA}/nn_hybrid_{TAG}.csv', index=False, encoding='utf-8')

if WRITE_ANSWER:
    if not os.path.exists(SPARSE_BACKUP) and os.path.exists(f'{DATA}/answer.csv'):
        shutil.copyfile(f'{DATA}/answer.csv', SPARSE_BACKUP)
    if HYBRID_VAL > SPARSE_VAL:
        shutil.copyfile(f'{DATA}/nn_hybrid_{TAG}.csv', f'{DATA}/answer.cs

chk = pd.read_csv(f'{DATA}/nn_hybrid_{TAG}.csv', dtype=str)
ans = chk.answer.fillna('').str.split()
assert list(chk.columns) == ['query_id', 'answer'] and len(chk) == len(bq) == chk.query_id.nunique()
assert set(chk.query_id) == set(bq.query_id) and ans.map(len).eq(TOPK).all()
assert ans.map(lambda l: len(set(l)) == len(l)).all()             # без дублей
assert not ans.map(lambda l: bool(set(l) - bench_ids)).any()   # все id из корпуса